# IMF WEO dataframe demo

This notebook shows a minimal `pysdmx` workflow for IMF WEO using the current SDMX 3.0 base URL. For WEO, the SDMX key is `COUNTRY.INDICATOR.FREQUENCY`. `UNIT` and `SCALE` are returned as attributes, so they are filtered after retrieval rather than placed in the query key.

Useful code fields in this repo:
- `COUNTRY`: WEO country code such as `GBR` or `USA`
- `INDICATOR`: WEO subject series code such as `NGDPD`
- `FREQUENCY`: usually `A` for annual WEO series
- `UNIT`: returned attribute such as `USD`
- `SCALE`: returned attribute such as `9` for billions


In [ ]:
import pandas as pd
from pysdmx.api.qb import ApiVersion, DataContext, DataFormat, DataQuery, RestService, StructureFormat
from pysdmx.io.reader import read_sdmx

service = RestService(
    'https://api.imf.org/external/sdmx/3.0',
    ApiVersion.V2_2_2,
    data_format=DataFormat.SDMX_CSV_2_1_0,
    structure_format=StructureFormat.SDMX_JSON_2_0_0,
    timeout=60,
)

def add_scaled_values(frame: pd.DataFrame) -> pd.DataFrame:
    scaled = frame.copy()
    scale_powers = pd.to_numeric(scaled['SCALE'], errors='coerce').fillna(0)
    scaled['OBS_VALUE_SCALED'] = scaled['OBS_VALUE'] / (10 ** scale_powers)
    return scaled


In [ ]:
country_code = 'GBR'
indicator_code = 'NGDPD'
frequency = 'A'

query = DataQuery(
    context=DataContext.DATAFLOW,
    agency_id='IMF.RES',
    resource_id='WEO',
    version='+',
    key=f"{country_code}.{indicator_code}.{frequency}",
    obs_dimension='TIME_PERIOD',
    attributes='dsd',
)

raw_csv = service.data(query).decode('utf-8')
normalized_csv = raw_csv.replace('STRUCTURE[;]', 'STRUCTURE', 1)
message = read_sdmx(normalized_csv, validate=False)
weo_frame = message.data[0].data.copy()

weo_frame['TIME_PERIOD'] = pd.to_numeric(weo_frame['TIME_PERIOD'], errors='coerce').astype('Int64')
weo_frame['OBS_VALUE'] = pd.to_numeric(weo_frame['OBS_VALUE'], errors='coerce')
weo_frame = add_scaled_values(weo_frame)

weo_frame[['COUNTRY', 'INDICATOR', 'UNIT', 'SCALE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_SCALED']].head()


In [ ]:
unit_code = 'USD'
scale_code = '9'

filtered = weo_frame.loc[
    (weo_frame['UNIT'] == unit_code) & (weo_frame['SCALE'] == scale_code),
    ['COUNTRY', 'INDICATOR', 'UNIT', 'SCALE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_SCALED'],
]

filtered.head()


In [ ]:
country_code = '*'
indicator_code = 'NGDP'
frequency = 'A'

query = DataQuery(
    context=DataContext.DATAFLOW,
    agency_id='IMF.RES',
    resource_id='WEO',
    version='+',
    key=f"{country_code}.{indicator_code}.{frequency}",
    obs_dimension='TIME_PERIOD',
    attributes='dsd',
)

raw_csv = service.data(query).decode('utf-8')
normalized_csv = raw_csv.replace('STRUCTURE[;]', 'STRUCTURE', 1)
message = read_sdmx(normalized_csv, validate=False)
all_ngdp_frame = message.data[0].data.copy()

all_ngdp_frame['TIME_PERIOD'] = pd.to_numeric(all_ngdp_frame['TIME_PERIOD'], errors='coerce').astype('Int64')
all_ngdp_frame['OBS_VALUE'] = pd.to_numeric(all_ngdp_frame['OBS_VALUE'], errors='coerce')
all_ngdp_frame = add_scaled_values(all_ngdp_frame)


In [ ]:
national_gdp = all_ngdp_frame.loc[
    (all_ngdp_frame['UNIT'] == 'XDC') & (all_ngdp_frame['SCALE'] == '9'),
    ['COUNTRY', 'INDICATOR', 'UNIT', 'SCALE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_SCALED'],
]

national_gdp.sort_values(['COUNTRY', 'TIME_PERIOD']).head(20)


In [ ]:
country_code = '*'
indicator_code = 'NGDPD'
frequency = 'A'

query = DataQuery(
    context=DataContext.DATAFLOW,
    agency_id='IMF.RES',
    resource_id='WEO',
    version='+',
    key=f"{country_code}.{indicator_code}.{frequency}",
    obs_dimension='TIME_PERIOD',
    attributes='dsd',
)

raw_csv = service.data(query).decode('utf-8')
normalized_csv = raw_csv.replace('STRUCTURE[;]', 'STRUCTURE', 1)
message = read_sdmx(normalized_csv, validate=False)
all_ngdpd_frame = message.data[0].data.copy()

all_ngdpd_frame['TIME_PERIOD'] = pd.to_numeric(all_ngdpd_frame['TIME_PERIOD'], errors='coerce').astype('Int64')
all_ngdpd_frame['OBS_VALUE'] = pd.to_numeric(all_ngdpd_frame['OBS_VALUE'], errors='coerce')
all_ngdpd_frame = add_scaled_values(all_ngdpd_frame)


In [ ]:
all_ngdpd = all_ngdpd_frame.loc[
    (all_ngdpd_frame['UNIT'] == 'USD') & (all_ngdpd_frame['SCALE'] == '9'),
    ['COUNTRY', 'INDICATOR', 'UNIT', 'SCALE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_SCALED'],
]

all_ngdpd.sort_values(['COUNTRY', 'TIME_PERIOD']).head(20)
